In [ ]:
#
# PURPOSE:
# 1. Load all raw flight data from CSV files.
# 2. Process, clean, and convert the data into time series.
# 3. Save the final, processed data (Healthy, Faulty, Test) into a single
#    Python pickle file (.pkl) for all other notebooks to use.

import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# --- 1. Configuration ---

# Set the main output directory for all project artifacts
# This notebook will place its output here.
OUT_DIR = Path("Final_Run_Outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Define file paths
PKL_PATH = OUT_DIR / "phase1_prepared_data.pkl"
FLIGHT_DATA_PATH = "NGAFID" # Assumes 'NGAFID' folder is in the same directory
HEALTHY_FLIGHTS_DIR = f"{FLIGHT_DATA_PATH}/Healthy Flights"
MAINTENANCE_FLIGHTS_DIR = f"{FLIGHT_DATA_PATH}/Maintenance Flights"
TEST_FLIGHTS_DIR = f"{FLIGHT_DATA_PATH}/Test Flights"
TEST_LABELS_PATH = f"{TEST_FLIGHTS_DIR}/test_labels.csv"

# Define the columns (sensors) we want to extract from the CSVs
COLUMNS_TO_USE = [
    'AltB', 'AltMSL', 'BaroA', 'E1_EGT_Max', 'E1_Eps', 'E1_FF_Max',
    'E1_N1_Max', 'E1_N2_Max', 'E1_OILP_Max', 'E1_OILT_Max',
    'E1_Selected_Eng', 'E1_VIB_Max', 'FQtyL', 'FQtyR', 'GndSpd',
    'IAS', 'LatAc', 'NormAc', 'OAT', 'Pitch', 'Roll', 'TAS', 'VSpd'
]

print("Configuration set. Output directory created.")


# --- 2. Data Loading Function ---

def load_flight_data(directory, file_limit=None):
    """Loads all CSVs from a directory into a list of DataFrames."""
    flight_list = []
    paths = sorted(list(Path(directory).glob("*.csv")))
    
    if file_limit:
        paths = paths[:file_limit]
        
    print(f"Loading {len(paths)} files from {directory}...")
    
    for file_path in paths:
        try:
            df = pd.read_csv(file_path, usecols=COLUMNS_TO_USE)
            if not df.empty:
                flight_list.append(df.values.astype(np.float32))
        except Exception as e:
            print(f"Could not read {file_path}: {e}")
            
    return flight_list

# --- 3. Execute Loading and Processing ---

# Load Healthy Flights
# We limit to 8736 as this was the number used in your successful classifier
healthy_timeseries = load_flight_data(HEALTHY_FLIGHTS_DIR, file_limit=8736)

# Load Faulty Flights (these are the 30 "pre-maintenance" flights)
faulty_timeseries_for_gan = load_flight_data(MAINTENANCE_FLIGHTS_DIR, file_limit=30)

# Load Test Flights
test_timeseries = load_flight_data(TEST_FLIGHTS_DIR)
test_labels_df = pd.read_csv(TEST_LABELS_PATH)
test_labels = test_labels_df['label'].values

print(f"\nLoaded {len(healthy_timeseries)} healthy series.")
print(f"Loaded {len(faulty_timeseries_for_gan)} faulty series.")
print(f"Loaded {len(test_timeseries)} test series.")


# --- 4. Create and Save Pickle File ---

# Combine all loaded data into a single dictionary
data_to_save = {
    "healthy_timeseries": healthy_timeseries,
    "faulty_timeseries_for_gan": faulty_timeseries_for_gan,
    "test_timeseries": test_timeseries,
    "test_labels": test_labels
}

# Save the dictionary to a pickle file
with open(PKL_PATH, "wb") as f:
    pickle.dump(data_to_save, f)

print(f"\nData preparation complete.")
print(f"All data saved to: {PKL_PATH.resolve()}")
print("\n--- You may now proceed to Notebook 02 ---")